# Multi-Agent Socio-Physical Simulation

**SOTA Techniques:** Mesa Agent-Based Modeling, Swarm Intelligence, Cellular Automata, Emergence Detection

---

## Overview

Advanced multi-agent simulation with emergent behavior analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

## 1. Load Agent State Data

In [ ]:
data_dir = '../data/synthetic'
try:
    df = pd.read_csv(f'{data_dir}/agent_states.csv')
    print(f'Loaded {len(df)} agent state records')
except:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'agent_id': [f'AGT{i:04d}' for i in range(n)],
        'agent_type': np.random.choice(['pedestrian', 'vehicle', 'service'], n),
        'x_pos': np.random.uniform(0, 100, n),
        'y_pos': np.random.uniform(0, 100, n),
        'state': np.random.choice(['moving', 'idle', 'interacting'], n),
        'interaction_count': np.random.poisson(3, n),
        'speed': np.random.uniform(0, 5, n)
    })
    print(f'Created {len(df)} synthetic agent states')
print(df.head())

## 2. Spatial Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Position scatter
axes[0].scatter(df['x_pos'], df['y_pos'], c=df['agent_type'].map({'pedestrian':0,'vehicle':1,'service':2}),
                cmap='viridis', alpha=0.6)
axes[0].set_xlabel('X Position')
axes[0].set_ylabel('Y Position')
axes[0].set_title('Agent Spatial Distribution')
axes[0].grid(True, alpha=0.3)

# State distribution
state_counts = df['state'].value_counts()
axes[1].bar(state_counts.index, state_counts.values)
axes[1].set_ylabel('Count')
axes[1].set_title('State Distribution')
axes[1].tick_params(axis='x', rotation=45)

# Speed distribution
axes[2].hist(df['speed'], bins=30, alpha=0.7)
axes[2].set_xlabel('Speed')
axes[2].set_ylabel('Count')
axes[2].set_title('Speed Distribution')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Interaction Network Analysis

In [ ]:
# Build interaction graph
interaction_graph = defaultdict(list)
for _, row in df[df['interaction_count'] > 0].iterrows():
    for i in range(int(row['interaction_count'])):
        target_id = f'AGT{(int(row["agent_id"][3:]) + i + 1) % 1000:04d}'
        interaction_graph[row['agent_id']].append(target_id)

print(f'Agents with interactions: {len(interaction_graph)}')
print(f'Average interactions per agent: {sum(len(v) for v in interaction_graph.values())/len(interaction_graph):.2f}')

# Visualize top interactors
top_agents = sorted(interaction_graph.items(), key=lambda x: len(x[1]), reverse=True)[:10]
top_ids = [a[0] for a in top_agents]
top_counts = [len(a[1]) for a in top_agents]

plt.figure(figsize=(10, 4))
plt.bar(range(len(top_ids)), top_counts)
plt.xlabel('Agent Rank')
plt.ylabel('Interaction Count')
plt.title('Top 10 Most Interactive Agents')
plt.xticks(range(len(top_ids)), top_ids, rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Emergent Pattern Detection

Detect clusters and patterns in agent behavior.

In [ ]:
# K-means clustering on positions
from sklearn.cluster import KMeans

X = df[['x_pos', 'y_pos']].values
kmeans = KMeans(n_clusters=4, random_state=42)
df['cluster'] = kmeans.fit_predict(X)

# Visualize clusters
plt.figure(figsize=(8, 6))
for cluster in range(4):
    cluster_data = df[df['cluster'] == cluster]
    plt.scatter(cluster_data['x_pos'], cluster_data['y_pos'],
                label=f'Cluster {cluster}', alpha=0.6, s=50)

plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c='red', marker='X', s=200, label='Centroids')
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Emergent Spatial Clusters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Cluster centers:\n{kmeans.cluster_centers_}')

## 5. Cellular Automata Simulation

Simple CA model for emergent behavior.

In [ ]:
def cellular_automaton(grid, rules, steps):
    """Simple 2D CA with Moore neighborhood."""
    result = [grid[:]]
    for _ in range(steps):
        new_grid = [row[:] for row in grid]
        for i in range(1, len(grid)-1):
            for j in range(1, len(grid[0])-1):
                # Count active neighbors
                neighbors = sum(grid[i+di][j+dj] for di in [-1,0,1] for dj in [-1,0,1] if (di,dj)!=(0,0))
                new_grid[i][j] = rules[grid[i][j]][neighbors]
        result.append(new_grid)
    return result

# Initialize grid
grid_size = 20
grid = [[0 if np.random.random() > 0.3 else 1 for _ in range(grid_size)] for _ in range(grid_size)]

# Conway-like rules
rules = {
    0: lambda n: 1 if n == 3 else 0,  # Birth
    1: lambda n: 1 if n in [2, 3] else 0  # Survival
}

result = cellular_automaton(grid, rules, 10)

plt.figure(figsize=(12, 4))
for i, g in enumerate(result[::2]):  # Every other step
    plt.subplot(1, 5, i+1)
    plt.imshow(g, cmap='YlGnBu')
    plt.title(f'Step {i*2}')
    plt.axis('off')

plt.tight_layout()
plt.show()

## 6. Swarm Behavior Analysis

Analyze collective motion patterns.

In [ ]:
# Calculate velocity vectors from position changes
df_sorted = df.sort_values('agent_id')
velocities = []
for agent_id in df['agent_id'].unique():
    agent_data = df_sorted[df_sorted['agent_id'] == agent_id].sort_index()
    if len(agent_data) > 1:
        dx = agent_data['x_pos'].diff().dropna().mean()
        dy = agent_data['y_pos'].diff().dropna().mean()
        velocities.append([dx, dy])

if velocities:
    velocities = np.array(velocities)
    
    plt.figure(figsize=(6, 6))
    plt.quiver(df['x_pos'].iloc[::10], df['y_pos'].iloc[::10], 
               velocities[::10, 0], velocities[::10, 1], 
               scale=50, color='steelblue', alpha=0.7)
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.title('Agent Velocity Vectors')
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    plt.show()
    
    print(f'Mean velocity: {velocities.mean(axis=0)}')
    print(f'Velocity std: {velocities.std(axis=0)}')

## Summary

This notebook demonstrated:
1. **Agent state analysis**
2. **Spatial distribution patterns**
3. **Interaction network analysis**
4. **Emergent cluster detection**
5. **Cellular automata modeling**
6. **Swarm behavior analysis**